In [186]:
import sys
import numpy as np
from scipy.optimize import linprog
import itertools
import os
import time
from functools import partial

In [187]:
import random
import math

In [188]:
random.seed(42)

In [189]:
tests = ['fl_25_2', 'fl_100_1', 'fl_200_7', 'fl_500_7', 'fl_1000_2', 'fl_2000_2']
thresholds = [(4000000, 3269822), (26000000, 22724634), (5000000, 4711295), (30000000, 27006099), (10000000, 8879294), (10000000, 7453531)]

In [190]:
def load_data(test):
    with open(f"data/{test}") as file:
        lines = file.readlines()
        n, m = list(map(int, lines[0].split()))
        price = list()
        cap = list()
        shops = list()
        for i in range(n):
            p, c, x, y = list(map(float, lines[1 + i].split()))
            price.append(p)
            cap.append(c)
            shops.append((x, y))
            
        calls = list()
        customers = list()
        for i in range(m):
            call, x, y = list(map(float, lines[1 + n + i].split()))
            calls.append(call)
            customers.append((x, y))
            
        return n, m, price, cap, shops, calls, customers

Я не понял, почему здесь нужно как - то формулировать задачу, ибо вроде здесь и так понятна формальная постановка:

Нужно выбрать множество магазинов $S$ и $F: [M] \rightarrow S$, которая будет задавать для каждого покупателя магазин, в который он будет ходить, причем минимизация будет происходить по $$\sum_{i \in S} s_i + \sum_{i=1}^{M} \text{dist}(\text{customer}_i, \text{shop}_{F_i})$$

Требуется выполнить условия:
$$\forall i \in [N]: \sum_{x \in F^{-1}(i)} d_x \leq cap_i$$

In [191]:
def check_facility(n, m, price, cap, shops, calls, customers, choise, f):
    if len(set(choise)) != len(choise):
        raise Exception("Not correct facility")

    if len(choise) == 0:
        raise Exception("Not correct facility")

    if min(choise) < 0 or max(choise) >= n:
        raise Exception("Not correct facility")

    result = 0
    for i in choise:
        result += price[i]

    sum_cap = [0] * n
    for i in range(m):
        shop = choise[f[i]]
        sum_cap[shop] += calls[i]
        
        result += math.dist(customers[i], shops[shop])

    for i in range(n):
        if sum_cap[i] > cap[i]:
            raise Exception("Not correct facility")

    return result

In [192]:
def passed_cnt(result, idx):
    if result <= thresholds[idx][1]:
        return 2
    elif result <= thresholds[idx][0]:
        return 1
    else:
        return 0

In [193]:
def test_method(method, name, use_file=False):
    print(f"Checking {name}")
    score = 0
    for i, test in enumerate(tests):
        n, m, price, cap, shops, calls, customers = load_data(test)
        start = time.time()
        
        if not use_file:
            choise, f = method(n, m, price, cap, shops, calls, customers)
        else:
            choise, f = method(test)

        end = time.time()
        elapsed = end - start
        print(f"Execution time: {elapsed:.4f} seconds")
            
        result = check_facility(n, m, price, cap, shops, calls, customers, choise, f)
        passed = passed_cnt(result, i)
        
        if passed == 1:
            score += 3
        elif passed == 2:
            score += 5

        print(f"Target function {test}: {result}")
        print(f"Passed {test}: {passed}")

    print(f"Score: {score}")

Идея:
\
Пусть множество магазинов фиксированно. Подберем выбор покупателям из следующих жадных соображений:
в порядке убывания их $d_c$ будем назначать их в магазин с минимальным расстоянием, чтобы это не превышало $\text{cap}$ магазина
\
Множество магазинов выбирать будем таким образом: берем случайный порядок магазинов и добавляем магазин, пока не получится распределить всех покупателей. В качестве инициализации возьмем порядок по возрастанию стоимости открытия.
\
Дальше будем работать с лучшим решением. Удаляем случайное подмножество магазинов (долю <= ratio) и добавляем в случайном порядке магазины, пока решение не станет валидным. Итого выходят жадные эвристики c выкидыванием решения и его достройкой.

In [205]:
!g++ -O2 -std=c++2a cpp_methods/greedy.cpp -o tmp/greedy

In [206]:
def greedy_facility(test_file, ratio): 
    os.system(f"./tmp/greedy data/{test_file} {ratio}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        choice = list(map(int, lines[0].split()))
        f = list(map(int, lines[1].split()))
        return choice, f

In [207]:
test_method(partial(greedy_facility, ratio=0.01), "greedy", True)

Checking greedy
Execution time: 60.3157 seconds
Target function fl_25_2: 3336471.400341548
Passed fl_25_2: 1
Execution time: 60.0110 seconds
Target function fl_100_1: 23638312.62870556
Passed fl_100_1: 1
Execution time: 60.0103 seconds
Target function fl_200_7: 4875471.959699263
Passed fl_200_7: 1
Execution time: 60.2857 seconds
Target function fl_500_7: 32182250.914409522
Passed fl_500_7: 0
Execution time: 60.1679 seconds
Target function fl_1000_2: 11266435.28433089
Passed fl_1000_2: 0
Execution time: 61.9862 seconds
Target function fl_2000_2: 10807027.384238105
Passed fl_2000_2: 0
Score: 9


In [208]:
test_method(partial(greedy_facility, ratio=0.1), "greedy", True)

Checking greedy
Execution time: 60.0044 seconds
Target function fl_25_2: 3298971.400341548
Passed fl_25_2: 1
Execution time: 60.0123 seconds
Target function fl_100_1: 23429270.95363265
Passed fl_100_1: 1
Execution time: 60.0198 seconds
Target function fl_200_7: 4964299.654752965
Passed fl_200_7: 1
Execution time: 60.2330 seconds
Target function fl_500_7: 32390236.873675235
Passed fl_500_7: 0
Execution time: 60.1333 seconds
Target function fl_1000_2: 11493283.578673692
Passed fl_1000_2: 0
Execution time: 62.4647 seconds
Target function fl_2000_2: 10689097.170332758
Passed fl_2000_2: 0
Score: 9


In [209]:
test_method(partial(greedy_facility, ratio=0.2), "greedy", True)

Checking greedy
Execution time: 60.0039 seconds
Target function fl_25_2: 3269821.3205308816
Passed fl_25_2: 2
Execution time: 60.0091 seconds
Target function fl_100_1: 23429270.95363265
Passed fl_100_1: 1
Execution time: 60.0223 seconds
Target function fl_200_7: 4973200.212848658
Passed fl_200_7: 1
Execution time: 60.3097 seconds
Target function fl_500_7: 33007047.706769083
Passed fl_500_7: 0
Execution time: 60.2895 seconds
Target function fl_1000_2: 11396006.334700817
Passed fl_1000_2: 0
Execution time: 62.3397 seconds
Target function fl_2000_2: 10762287.471350066
Passed fl_2000_2: 0
Score: 11


In [216]:
test_method(partial(greedy_facility, ratio=0.3), "greedy", True)

Checking greedy
Execution time: 60.0055 seconds
Target function fl_25_2: 3269821.3205308816
Passed fl_25_2: 2
Execution time: 60.0095 seconds
Target function fl_100_1: 23422334.798554894
Passed fl_100_1: 1
Execution time: 60.0166 seconds
Target function fl_200_7: 5022323.346888955
Passed fl_200_7: 0
Execution time: 60.2090 seconds
Target function fl_500_7: 33768268.04913876
Passed fl_500_7: 0
Execution time: 60.2231 seconds
Target function fl_1000_2: 11433468.379384635
Passed fl_1000_2: 0
Execution time: 62.9444 seconds
Target function fl_2000_2: 10605782.945896454
Passed fl_2000_2: 0
Score: 8


Мы умеем понятным образом сравнивать допустимые решения, но недопустимые мы пока никак не различаем. Предлагаю ввести штраф за каждую единицу превышения ограничения магазина $\lambda$. Тут я хочу попробовать не привязываться к пайплайну выбора магазинов с жадным выбором, куда идти клиенту. Я буду считать, что каждому клиенту сопоставлен магазин и я буду менять выборы клиентов. 

Будем делать локальные оптимизации двух видов:
1. Выбрать клиенту оптимальный по целевой функции магазин.
2. Поменять выборы двух клиентов местами. Вторую операцию будем делать, только если не получилось добиться улучшения первой операцией.

Тогда построим решение, как генерация случайного решения (возможно невалидного), а потом применение оптимизаций этих двух видов, пока получается. Добавим рестарты.

In [217]:
!g++ -O2 -std=c++2a cpp_methods/local_opt.cpp -o tmp/local_opt

In [218]:
def local_opt_facility(test_file): 
    os.system(f"./tmp/local_opt data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        choice = list(map(int, lines[0].split()))
        f = list(map(int, lines[1].split()))
        return choice, f

In [219]:
test_method(local_opt_facility, "local_opt", True)

Checking local_opt
Execution time: 60.8066 seconds
Target function fl_25_2: 3269821.3205308816
Passed fl_25_2: 2
Execution time: 60.1215 seconds
Target function fl_100_1: 135552771.45272258
Passed fl_100_1: 0
Execution time: 60.0277 seconds
Target function fl_200_7: 5011546.748513756
Passed fl_200_7: 0
Execution time: 63.0747 seconds
Target function fl_500_7: 35906604.86047985
Passed fl_500_7: 0
Execution time: 63.1853 seconds
Target function fl_1000_2: 9742225.910342462
Passed fl_1000_2: 1
Execution time: 60.5765 seconds
Target function fl_2000_2: 8655397.252881745
Passed fl_2000_2: 1
Score: 11


Довольно странные результаты получились.

Вернемся к идее, что для фиксированного множества магазинов задача будет решаться жадно и нужно подобрать хороший набор магазинов. Потому что, видимо, менять функцию сопоставления маленькими действиями неэффективно. 
\
Применим идею со штрафами чуть другим методом. Мы умеем понятным образом сравнивать множества допустимых магазинов, но недопустимые множества мы пока никак не различаем. Аналогично хочу ввести штраф, но за каждого необработанного клиента $\lambda$. Теперь все множества сравнимы численно. 
Сделаем отжиг на множество магазинов, где изменением будет 1-flip (добавление/удаление 1 магазина).

Я планирую перебрать шаг изменения температуры в отжиге, а $\lambda$ не планирую, так как хочу его взять достаточно большим, чтобы недопустимое решение было по весу больше слабого порога и дальнейшего смысла в его подгоне нет.

In [210]:
!g++ -O2 -std=c++2a cpp_methods/annealing.cpp -o tmp/annealing

In [211]:
def sa_facility(test_file, step_temp=0.9): 
    os.system(f"./tmp/annealing data/{test_file} {step_temp}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        choice = list(map(int, lines[0].split()))
        f = list(map(int, lines[1].split()))
        return choice, f

In [212]:
test_method(partial(sa_facility, step_temp=0.9), "sa", True)

Checking sa
Execution time: 60.3968 seconds
Target function fl_25_2: 3269821.3205308816
Passed fl_25_2: 2
Execution time: 60.1378 seconds
Target function fl_100_1: 23926449.8417158
Passed fl_100_1: 1
Execution time: 60.1450 seconds
Target function fl_200_7: 4855341.559466326
Passed fl_200_7: 1
Execution time: 62.7929 seconds
Target function fl_500_7: 28581900.399229422
Passed fl_500_7: 1
Execution time: 65.5346 seconds
Target function fl_1000_2: 9442408.078446282
Passed fl_1000_2: 1
Execution time: 65.0553 seconds
Target function fl_2000_2: 8162126.2646327205
Passed fl_2000_2: 1
Score: 20


In [213]:
test_method(partial(sa_facility, step_temp=0.99), "sa", True)

Checking sa
Execution time: 60.0084 seconds
Target function fl_25_2: 3269821.3205308816
Passed fl_25_2: 2
Execution time: 60.0101 seconds
Target function fl_100_1: 23926449.8417158
Passed fl_100_1: 1
Execution time: 60.1571 seconds
Target function fl_200_7: 4859371.1290985355
Passed fl_200_7: 1
Execution time: 63.5136 seconds
Target function fl_500_7: 28581900.399229422
Passed fl_500_7: 1
Execution time: 60.1378 seconds
Target function fl_1000_2: 9445203.909808319
Passed fl_1000_2: 1
Execution time: 68.3612 seconds
Target function fl_2000_2: 8162126.2646327205
Passed fl_2000_2: 1
Score: 20


In [214]:
test_method(partial(sa_facility, step_temp=0.999), "sa", True)

Checking sa
Execution time: 60.0053 seconds
Target function fl_25_2: 3269821.3205308816
Passed fl_25_2: 2
Execution time: 60.0879 seconds
Target function fl_100_1: 23926449.8417158
Passed fl_100_1: 1
Execution time: 60.3451 seconds
Target function fl_200_7: 4868423.581687911
Passed fl_200_7: 1
Execution time: 64.6403 seconds
Target function fl_500_7: 28579105.55648339
Passed fl_500_7: 1
Execution time: 60.1732 seconds
Target function fl_1000_2: 9383296.562330611
Passed fl_1000_2: 1
Execution time: 63.1198 seconds
Target function fl_2000_2: 8149920.570414765
Passed fl_2000_2: 1
Score: 20


In [215]:
test_method(partial(sa_facility, step_temp=0.9999), "sa", True)

Checking sa
Execution time: 60.0057 seconds
Target function fl_25_2: 3269821.3205308816
Passed fl_25_2: 2
Execution time: 60.1289 seconds
Target function fl_100_1: 23926449.8417158
Passed fl_100_1: 1
Execution time: 60.0535 seconds
Target function fl_200_7: 4860431.32327108
Passed fl_200_7: 1
Execution time: 62.6681 seconds
Target function fl_500_7: 28633482.474659417
Passed fl_500_7: 1
Execution time: 60.0210 seconds
Target function fl_1000_2: 9420781.48199693
Passed fl_1000_2: 1
Execution time: 63.3748 seconds
Target function fl_2000_2: 8155660.229228312
Passed fl_2000_2: 1
Score: 20


Теперь прошли все простые пороги и один сложный

Итог
1. Первым решением, которое дало баллы стала комбинация жадных эвристик и LNS. Сводим задачу к выбору набора магазинов, а сопоставление курьеров происходит жадно. Поиск магазинов является двухфазным. Сначала мы находим лучшее решение из решений вида: в случайном порядке магазинов берем магазины, пока не получится обработать всех клиентов. Дальше это решение сколько - то итераций разрушаем и достраиваем. При правильном подборе гиперпараметров оно дает 11 баллов.
2. Я решил попробовать метод, который не будет опираться на структуру выбор магазинов + жадное сопоставление, а пытаюсь менять саму функцию сопоставления. Ввожу понятие штрафа за превышение ограничений магазинов и использую две эвристики (поменять магазин у клиента, поменять магазины 2 клиентов). Такое решение также дало 11 баллов, но тесты отличаются.
3. Вернулся к идее выбора магазинов, добавил систему штрафов другим методом. Поверх этого сделал метод отжига выбора магазинов с мутацией 1-flip. Итоговое решение получило 20 баллов.